# Clase 12 — Gobernanza, costes y LLMOps + proyecto

**Curso:** MLOps y Monitorización con Python · **Bloque B** · **ÚLTIMA clase**
**Tiempo estimado:** 120 minutos · **Capítulo del manual:** `manual/12_gobernanza_costes_llmops.md`

## Qué verás en esta clase

- **Gobernanza:** un pequeño registro de auditoría que une DVC (clase 3), MLflow (clase 4) y el registro de modelos con alias (clase 8) en un informe legible de trazabilidad completa.
- **Costes:** una estimación REAL, calculada en Python, del coste de entrenamiento, inferencia (apoyada en la latencia real de la clase 7) y almacenamiento (con tamaños reales de archivo) de un sistema en producción.
- **LLMOps:** en qué se parece y en qué se diferencia del MLOps clásico de este curso, con una cadena RAG simulada (sin ningún modelo de lenguaje real) trazada con `mlflow.trace`.
- **Proyecto integrador:** un caso nuevo y más pequeño (cáncer de mama, `sklearn.datasets.load_breast_cancer`) recorriendo con criterio las decisiones de las clases 2 a 11.

> **Nota:** todo ocurre dentro de una carpeta de trabajo efímera (`proyecto_gobernanza/`), creada al principio y **eliminada al final** de este mismo notebook.

> **Atención:** todas las cifras de coste son un *ejemplo ilustrativo*, con constantes declaradas explícitamente (coste por hora de CPU, coste por GB, coste por 1000 tokens). El punto pedagógico es la fórmula y el razonamiento, no el número absoluto.

## Prerequisitos

- De la clase 3: linaje de datos y modelos con DVC (hash, commit).
- De la clase 4: seguimiento de experimentos con MLflow.
- De la clase 8: registro de modelos con alias candidato/producción/archivado.
- De la clase 7: la latencia real medida del servicio de predicción.
- De las clases 2, 6, 10: semilla y config explícita, tests mínimos, control de umbrales.

> **Cómo usar este notebook:** ejecuta las celdas en orden, de arriba a abajo, una sola vez. Tarda menos de un minuto en total.

---
## Instalación

Instala solo lo que falte (incluye MLflow y joblib).

In [ ]:
import os
import warnings

# Silenciamos avisos de compatibilidad fijos (no aportan nada aquí) para que
# la salida de este notebook no dependa de detalles internos de versiones de librerías
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Igual que en las clases 4 y 8: fijamos estas variables ANTES de importar mlflow y subimos su logger a ERROR
# (WARNING no basta: mlflow.utils.requirements_utils avisa en WARNING con detalles no deterministicos)
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import subprocess
import sys


def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("joblib", "joblib")
asegurar("mlflow", "mlflow")
print("Librerías listas.")

---
## Celda 0 — Setup

In [ ]:
import os
import re
import shutil
import json
import logging
import warnings

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.datasets import load_breast_cancer

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

import mlflow

logging.getLogger("mlflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=DeprecationWarning)

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
COLUMNAS_ENTRADA = COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS

print("Librerías y funciones auxiliares listas. Versión de MLflow:", mlflow.__version__)

---
## Carpeta de trabajo efímera

In [ ]:
CARPETA = "proyecto_gobernanza"

# Limpiamos y recreamos la carpeta para que el notebook sea repetible desde cero (como en todas las clases anteriores)
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(CARPETA)
print("Carpeta de trabajo creada:", os.path.abspath(CARPETA))

---
## 1. Gobernanza: un registro de auditoría de trazabilidad completa

Unimos, en un solo informe, lo que ya existe por separado: el linaje de datos y modelos (clase 3, hashes de DVC), el seguimiento de experimentos (clase 4, `run_id` de MLflow) y el registro de modelos con alias (clase 8, candidato/producción/archivado). Los eventos de abajo son **simulados** (no una ejecución real de esas herramientas), pero la función que los procesa es código real.

In [ ]:
# Eventos SIMULADOS (datos de ejemplo, no una ejecución real de DVC/MLflow/registro): unen conceptualmente
# el linaje de datos (clase 3), el seguimiento de experimentos (clase 4) y el registro de modelos con alias (clase 8)
EVENTOS = [
    {"tipo": "version_datos", "version_modelo": "v1", "actor": "ana_datos", "cuando": "2026-01-10",
     "detalle": "hash_dvc=a1b2c3d4 (churn.csv, clase 3)"},
    {"tipo": "entrenamiento", "version_modelo": "v1", "actor": "ana_datos", "cuando": "2026-01-11",
     "detalle": "run_mlflow=r001, class_weight=ninguno, f1=0.02 (clase 4)"},
    {"tipo": "promocion_candidato", "version_modelo": "v1", "actor": "ana_datos", "cuando": "2026-01-11",
     "detalle": "alias candidato -> v1 (clase 8)"},
    {"tipo": "aprobacion_produccion", "version_modelo": "v1", "actor": "luis_revisor", "cuando": "2026-01-12",
     "detalle": "alias produccion -> v1, tests superados (clase 6)"},
    {"tipo": "version_datos", "version_modelo": "v2", "actor": "beto_ml", "cuando": "2026-02-03",
     "detalle": "hash_dvc=e5f6a7b8 (churn.csv v2, clase 3)"},
    {"tipo": "entrenamiento", "version_modelo": "v2", "actor": "beto_ml", "cuando": "2026-02-04",
     "detalle": "run_mlflow=r002, class_weight=balanced, f1=0.3166 (clase 4)"},
    {"tipo": "promocion_candidato", "version_modelo": "v2", "actor": "beto_ml", "cuando": "2026-02-04",
     "detalle": "alias candidato -> v2 (clase 8)"},
    {"tipo": "aprobacion_produccion", "version_modelo": "v2", "actor": "luis_revisor", "cuando": "2026-02-05",
     "detalle": "alias produccion -> v2, alias archivado -> v1 (clase 8)"},
]
print("Eventos de auditoría cargados:", len(EVENTOS))

In [ ]:
def generar_informe_auditoria(eventos):
    """Agrupa los eventos por versión de modelo y produce un informe legible de trazabilidad completa:
    quién entrenó, quién promovió, quién aprobó -- respondiendo "quién hizo qué y cuándo" para cada versión."""
    versiones = sorted({e["version_modelo"] for e in eventos})
    lineas = []
    for version in versiones:
        eventos_version = [e for e in eventos if e["version_modelo"] == version]
        lineas.append(f"Versión {version}:")
        for e in sorted(eventos_version, key=lambda x: x["cuando"]):
            lineas.append(f"  [{e['cuando']}] {e['tipo']} -- {e['actor']} :: {e['detalle']}")
    return "\n".join(lineas)


def verificar_separacion_de_funciones(eventos):
    """Principio de control de acceso (separación de funciones): quien ENTRENA una versión no debería
    ser la MISMA persona que la APRUEBA para producción. Devuelve la lista de versiones que violan esta regla."""
    violaciones = []
    for version in {e["version_modelo"] for e in eventos}:
        entrenadores = {e["actor"] for e in eventos if e["version_modelo"] == version and e["tipo"] == "entrenamiento"}
        aprobadores = {e["actor"] for e in eventos if e["version_modelo"] == version and e["tipo"] == "aprobacion_produccion"}
        if entrenadores & aprobadores:
            violaciones.append(version)
    return violaciones


informe = generar_informe_auditoria(EVENTOS)
print(informe)

RUTA_INFORME = os.path.join(CARPETA, "informe_auditoria.txt")
with open(RUTA_INFORME, "w") as f:
    f.write(informe)
print("\nInforme escrito en:", RUTA_INFORME)

violaciones = verificar_separacion_de_funciones(EVENTOS)
print("Versiones que violan la separación de funciones (entrenador == aprobador):", violaciones)
assert violaciones == [], "el registro simulado no debería tener violaciones de separación de funciones"
print("Separación de funciones respetada: nadie se aprobó a sí mismo.")

**¿Qué observas?** El informe responde, para cada versión, "quién entrenó", "quién promovió a candidato" y "quién aprobó producción" -- exactamente los tres elementos de trazabilidad completa que el capítulo describe. La función `verificar_separacion_de_funciones` codifica un principio de **control de acceso** (sección 2 del capítulo): quien entrena no debería ser quien aprueba su propio trabajo para producción, sin importar qué proveedor de nube o herramienta concreta se use.

---
## 2. Costes: una estimación real, calculada en Python

Tres componentes, cada uno con su fórmula explícita y sus constantes declaradas como *ejemplo ilustrativo*.

In [ ]:
# 2.1 Coste de entrenamiento: horas de cómputo ILUSTRATIVAS x un coste horario ILUSTRATIVO
# (el entrenamiento real de este proyecto tarda segundos en un portátil: las 2 horas simulan un caso de
# producción mayor, con más datos y validación cruzada -- el PUNTO es la fórmula, no la cifra absoluta)
HORAS_ENTRENAMIENTO_ILUSTRATIVO = 2.0
COSTO_HORA_CPU_ILUSTRATIVO = 0.05  # USD/hora, ejemplo ilustrativo, NO ligado a un proveedor concreto (los precios cambian)

costo_entrenamiento = round(HORAS_ENTRENAMIENTO_ILUSTRATIVO * COSTO_HORA_CPU_ILUSTRATIVO, 4)
print("Coste de entrenamiento (ejemplo ilustrativo):", costo_entrenamiento, "USD")

# 2.2 Coste de inferencia: se apoya en la latencia REAL medida en la clase 7 (allí, ≈4 ms por petición,
# valor usado también en su pregunta de autoevaluación 2) -- aquí la fijamos como constante para que el
# cálculo sea reproducible byte a byte (la clase 7 ya declaró que el milisegundo exacto varía por máquina)
LATENCIA_MS_ILUSTRATIVA = 4.0  # mismo orden de magnitud que la latencia real medida en la clase 7

costo_por_segundo = COSTO_HORA_CPU_ILUSTRATIVO / 3600
costo_por_prediccion = costo_por_segundo * (LATENCIA_MS_ILUSTRATIVA / 1000)
costo_por_1000_predicciones = round(costo_por_prediccion * 1000, 6)
print("Coste por 1000 predicciones (ejemplo ilustrativo):", costo_por_1000_predicciones, "USD")

PREDICCIONES_MES_ILUSTRATIVO = 100_000
costo_inferencia_mensual = round(costo_por_1000_predicciones * (PREDICCIONES_MES_ILUSTRATIVO / 1000), 4)
print("Coste de inferencia mensual, para", PREDICCIONES_MES_ILUSTRATIVO, "predicciones/mes (ejemplo ilustrativo):", costo_inferencia_mensual, "USD")

In [ ]:
# 2.3 Coste de almacenamiento: usamos el tamaño REAL (medido con os.path.getsize, no adivinado) del dataset
# (el mismo churn.csv de las clases 3-11) y del artefacto entrenado, x un coste por GB ILUSTRATIVO
RUTA_CSV = cargar_datos("churn.csv")
df = pd.read_csv(RUTA_CSV)
X = df[COLUMNAS_ENTRADA]
y = df["abandono"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEMILLA, stratify=y)

preprocesador = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
    ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
])
clasificador = LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000, C=1.0)
pipeline = Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador)])
pipeline.fit(X_train, y_train)
F1 = round(f1_score(y_test, pipeline.predict(X_test)), 4)
print("F1 del modelo reentrenado (config. ganadora de las clases 4/5/7/8):", F1)

RUTA_MODELO = os.path.join(CARPETA, "modelo.pkl")
joblib.dump(pipeline, RUTA_MODELO)

tamano_dataset_bytes = os.path.getsize(RUTA_CSV)
tamano_modelo_bytes = os.path.getsize(RUTA_MODELO)
tamano_total_gb = (tamano_dataset_bytes + tamano_modelo_bytes) / 1024 ** 3
print("Tamaño real de churn.csv (bytes):", tamano_dataset_bytes)
print("Tamaño real de modelo.pkl (bytes):", tamano_modelo_bytes)

COSTO_POR_GB_MES_ILUSTRATIVO = 0.02  # USD/GB/mes, ejemplo ilustrativo
costo_almacenamiento_mensual = round(tamano_total_gb * COSTO_POR_GB_MES_ILUSTRATIVO, 8)
print("Coste de almacenamiento mensual (ejemplo ilustrativo):", costo_almacenamiento_mensual, "USD")

costo_mensual_total = round(costo_entrenamiento + costo_inferencia_mensual + costo_almacenamiento_mensual, 4)
print("Coste TOTAL del primer mes (entrenamiento único + inferencia + almacenamiento, ejemplo ilustrativo):", costo_mensual_total, "USD")

**¿Qué observas?** El coste de entrenamiento (fijo, ocurre una vez) domina el primer mes de este ejemplo pequeño; el coste de inferencia escala con el volumen de predicciones (aquí, 100 000/mes ilustrativo); el coste de almacenamiento es prácticamente irrelevante para un dataset y un modelo de este tamaño (unos 154 KB en total) -- sería muy distinto con datasets de varios GB o modelos de miles de millones de parámetros, como los de la sección 3.

---
## 3. LLMOps: una cadena RAG simulada, trazada con MLflow

Ninguna llamada a un modelo de lenguaje real: "recuperar" y "generar" son funciones deterministas sobre una lista fija de 3 documentos. Las trazamos con `@mlflow.trace`, el mismo file store local de las clases 4 y 8 (logger a `ERROR`, sin servidor).

In [ ]:
# NO usamos ningún modelo de lenguaje real (sin API, sin descargas): "recuperación" y "generación" son
# funciones deterministas sobre una lista fija de documentos, para poder trazarlas con MLflow sin depender
# de ningún servicio externo
DOCUMENTOS = [
    {"id": "doc1", "texto": "El PSI mide el desplazamiento de una distribucion entre dos periodos usando buckets fijos"},  # dato-es
    {"id": "doc2", "texto": "MLflow registra experimentos modelos y ahora tambien trazas de llamadas encadenadas"},  # dato-es
    {"id": "doc3", "texto": "Docker empaqueta un servicio en una imagen reproducible con todas sus dependencias"},  # dato-es
]


def contar_tokens_falso(texto):
    """Conteo de tokens FALSO pero coherente (una palabra = un token), suficiente para ilustrar el
    principio de coste por token, sin tokenizador real."""
    return len(texto.split())


@mlflow.trace(span_type="RETRIEVER")
def recuperar(pregunta):
    """Paso 1: "recupera" el documento cuyo texto comparte más palabras con la pregunta (determinista)."""
    palabras_pregunta = set(pregunta.lower().split())
    mejor = max(DOCUMENTOS, key=lambda d: len(palabras_pregunta & set(d["texto"].lower().split())))
    return mejor


@mlflow.trace(span_type="LLM")
def generar(pregunta, documento):
    """Paso 2: "genera" una respuesta con una función mock (texto fijo, sin modelo real)."""
    return f"Segun {documento['id']}: {documento['texto']}"


@mlflow.trace(span_type="CHAIN")
def cadena_rag(pregunta):
    """Paso 3: encadena recuperación + generación -- la "cadena" que se traza como una unidad."""
    documento = recuperar(pregunta)
    return generar(pregunta, documento)


RUTA_MLRUNS = os.path.join(CARPETA, "mlruns")
mlflow.set_tracking_uri("file:" + RUTA_MLRUNS)
mlflow.set_experiment("rag_simulado")

PREGUNTA = "que mide el PSI"  # dato-es
with mlflow.start_run(run_name="consulta_rag"):
    respuesta = cadena_rag(PREGUNTA)
print("Pregunta:", PREGUNTA)
print("Respuesta:", respuesta)

### 3.1 Leer la traza guardada y mostrar su estructura (no sus tiempos)

In [ ]:
# Leemos la traza guardada de vuelta (file store local, misma cautela de las clases 4 y 8: logger a ERROR)
experimento = mlflow.get_experiment_by_name("rag_simulado")
trazas = mlflow.search_traces(experiment_ids=[experimento.experiment_id], flush=True)
assert len(trazas) == 1, "se esperaba exactamente una traza guardada"

traza = mlflow.get_trace(trazas.iloc[0]["trace_id"])
spans = traza.data.spans


def estructura_de_traza(spans):
    """Devuelve la ESTRUCTURA de la traza (nombres, tipos, anidamiento) sin ids ni tiempos absolutos --
    esos SÍ son deterministas, a diferencia de una duración real en milisegundos (la misma categoría de
    no-determinismo ya declarada en la clase 7 con la latencia real)."""
    raiz = next(s for s in spans if s.parent_id is None)
    hijos = [s for s in spans if s.parent_id == raiz.span_id]
    return {
        "n_spans": len(spans),
        "raiz": (raiz.name, raiz.span_type),
        "hijos_de_la_raiz": sorted((h.name, h.span_type) for h in hijos),
    }


ESTRUCTURA = estructura_de_traza(spans)
print("Estructura de la traza (n. de spans, nombres, tipos, anidamiento -- SIN tiempos absolutos):")
print(ESTRUCTURA)
assert ESTRUCTURA["n_spans"] == 3
assert ESTRUCTURA["raiz"] == ("cadena_rag", "CHAIN")
assert ESTRUCTURA["hijos_de_la_raiz"] == [("generar", "LLM"), ("recuperar", "RETRIEVER")]
print("La cadena tiene 3 pasos: 1 CHAIN raíz con 2 hijos (RETRIEVER y LLM) -- la misma forma en cualquier ejecución.")

> **Atención (determinismo entre lenguas):** una duración real en milisegundos de esta traza NO sería idéntica entre dos ejecuciones (la misma categoría de no-determinismo ya declarada en la clase 7 con la latencia real) -- por eso esta sección muestra la ESTRUCTURA de la traza (número de spans, nombres, tipos, anidamiento), que SÍ es idéntica en cualquier máquina y en cualquier ejecución.

### 3.2 Coste por token (ilustrativo), no por hora de cómputo

In [ ]:
# 3.3 Coste por token ILUSTRATIVO sobre la cadena simulada (coste por token, no por hora de cómputo --
# la diferencia clave del LLMOps frente al MLOps clásico, sección 3 del capítulo)
documento_recuperado = recuperar(PREGUNTA)
tokens_entrada = contar_tokens_falso(PREGUNTA) + contar_tokens_falso(documento_recuperado["texto"])
tokens_salida = contar_tokens_falso(respuesta)
tokens_totales = tokens_entrada + tokens_salida
print("Tokens de entrada (pregunta + documento recuperado):", tokens_entrada)
print("Tokens de salida (respuesta generada):", tokens_salida)

COSTO_POR_1000_TOKENS_ILUSTRATIVO = 0.002  # USD/1000 tokens, ejemplo ilustrativo, NO ligado a un proveedor concreto
costo_cadena = round((tokens_totales / 1000) * COSTO_POR_1000_TOKENS_ILUSTRATIVO, 8)
print("Coste de esta única llamada a la cadena (ejemplo ilustrativo):", costo_cadena, "USD")

### 3.3 Evaluar el texto generado: una métrica heurística, no una única cifra agregada

In [ ]:
# 3.4 Evaluar TEXTO generado: no hay un único número como F1 o AUC (clases 4-11) que baste por sí solo.
# Aquí usamos una métrica heurística DETERMINISTA (cobertura de palabras clave esperadas) solo como proxy
# ilustrativo -- un sistema real suele necesitar revisión humana o un segundo modelo "juez", no solo esto
def evaluar_respuesta(respuesta_generada, palabras_clave_esperadas):
    """Fracción de palabras clave esperadas que aparecen en la respuesta, entre 0 y 1."""
    palabras_respuesta = {p.strip(".,:;") for p in respuesta_generada.lower().split()}
    encontradas = palabras_clave_esperadas & palabras_respuesta
    return round(len(encontradas) / len(palabras_clave_esperadas), 4)


PALABRAS_CLAVE_ESPERADAS = {"psi", "desplazamiento", "distribucion"}
cobertura = evaluar_respuesta(respuesta, PALABRAS_CLAVE_ESPERADAS)
print("Cobertura de palabras clave esperadas en la respuesta:", cobertura)
assert cobertura == 1.0, "la respuesta debería contener las 3 palabras clave de este ejemplo"
print("La respuesta cubre el 100% de las palabras clave esperadas -- pero esta métrica NO detecta si el texto es coherente, natural o correcto en otros aspectos: solo mide una coincidencia léxica.")

**¿Qué observas?** La traza tiene siempre 3 spans con la misma forma (1 `CHAIN` raíz con 2 hijos, `RETRIEVER` y `LLM`), sin importar cuántas veces se ejecute -- es la diferencia con un tiempo real en milisegundos, que sí cambia. El coste se mide por TOKEN (entrada + salida), no por hora de cómputo reservada. Y evaluar el texto generado no es un F1 o un AUC (clases 4-11): la cobertura de palabras clave es solo un proxy heurístico, útil para un `assert` en este notebook, pero insuficiente por sí solo en un sistema real, donde hace falta revisión humana o un modelo "juez".

---
## 4. Proyecto integrador: un caso nuevo (cáncer de mama)

Recorremos, condensadas, las decisiones principales de las clases 2 a 11 sobre un dataset NUEVO y más pequeño que el churn: `sklearn.datasets.load_breast_cancer` (569 filas, 30 columnas numéricas, 2 clases, licencia libre).

In [ ]:
# 4.1 Config explícita + semilla fija (clase 2), sobre un caso NUEVO y más pequeño: cáncer de mama
# (scikit-learn, dominio público, clasificación binaria) -- NO es el proyecto de churn
CONFIG_PROYECTO_NUEVO = {
    "semilla": 42,
    "test_size": 0.25,
    "modelo": "LogisticRegression",
    "max_iter": 2000,
}

datos_cancer = load_breast_cancer()
X_cancer, y_cancer = datos_cancer.data, datos_cancer.target
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cancer, y_cancer, test_size=CONFIG_PROYECTO_NUEVO["test_size"],
    random_state=CONFIG_PROYECTO_NUEVO["semilla"], stratify=y_cancer,
)

pipeline_cancer = Pipeline([
    ("escalado", StandardScaler()),
    ("clasificador", LogisticRegression(max_iter=CONFIG_PROYECTO_NUEVO["max_iter"], random_state=CONFIG_PROYECTO_NUEVO["semilla"])),
])
pipeline_cancer.fit(Xc_train, yc_train)
pred_cancer = pipeline_cancer.predict(Xc_test)
proba_cancer = pipeline_cancer.predict_proba(Xc_test)[:, 1]

EXACTITUD_CANCER = round(accuracy_score(yc_test, pred_cancer), 4)
F1_CANCER = round(f1_score(yc_test, pred_cancer), 4)
AUC_CANCER = round(roc_auc_score(yc_test, proba_cancer), 4)
print("Dataset:", datos_cancer.data.shape, "filas x columnas, clases:", list(datos_cancer.target_names))
print("Exactitud (test):", EXACTITUD_CANCER, "| F1:", F1_CANCER, "| AUC:", AUC_CANCER)

In [ ]:
# 4.2 Un test mínimo de DATOS y uno de MODELO (clase 6), sin necesitar pandera ni una suite completa
def test_forma_de_los_datos():
    """Test de datos: 569 filas, 30 columnas, exactamente 2 clases -- lo que este capítulo espera de este dataset."""
    assert X_cancer.shape == (569, 30)
    assert set(np.unique(y_cancer)) == {0, 1}


def test_modelo_supera_umbral_minimo():
    """Test de modelo: como en la clase 6, el modelo no debe caer por debajo de un umbral conocido de antemano."""
    assert F1_CANCER >= 0.90


test_forma_de_los_datos()
test_modelo_supera_umbral_minimo()
print("2 tests superados: 1 de datos (forma y clases), 1 de modelo (F1 mínimo).")

In [ ]:
# 4.3 Una función de servicio simple (clase 7), sin FastAPI: predecir() con validación mínima
NOMBRES_COLUMNAS_CANCER = list(datos_cancer.feature_names)


def predecir(valores):
    """Recibe una lista de 30 números (mismo orden que NOMBRES_COLUMNAS_CANCER) y devuelve la probabilidad
    predicha de "benign" (clase 1). Valida longitud y tipo antes de llamar al modelo -- como la clase 7
    validaba con Pydantic, aquí a mano, sin esa dependencia."""
    if not isinstance(valores, (list, tuple)):
        raise TypeError("valores debe ser una lista o tupla")
    if len(valores) != len(NOMBRES_COLUMNAS_CANCER):
        raise ValueError(f"se esperaban {len(NOMBRES_COLUMNAS_CANCER)} valores, llegaron {len(valores)}")
    if not all(isinstance(v, (int, float)) for v in valores):
        raise TypeError("todos los valores deben ser numéricos")
    fila = np.array(valores).reshape(1, -1)
    probabilidad = float(pipeline_cancer.predict_proba(fila)[0, 1])
    return {"probabilidad_benigno": round(probabilidad, 4), "prediccion": int(probabilidad >= 0.5)}


ejemplo_valido = list(Xc_test[0])
resultado_prediccion = predecir(ejemplo_valido)
print("Predicción de ejemplo:", resultado_prediccion)

try:
    predecir([1.0, 2.0])  # longitud incorrecta -- debe fallar
    fallo_validacion = False
except ValueError:
    fallo_validacion = True
assert fallo_validacion
print("predecir() rechaza correctamente una entrada con longitud incorrecta.")

In [ ]:
# 4.4 Un control de monitorización/umbral (clase 10): comparar la exactitud de un lote reciente contra
# un umbral mínimo, avisando si cae por debajo
UMBRAL_EXACTITUD_MINIMA = 0.90


def revisar_umbral_calidad(exactitud_actual, umbral=UMBRAL_EXACTITUD_MINIMA):
    """Misma lógica que detectar_incidentes de la clase 10, aplicada a una sola métrica de modelo."""
    return exactitud_actual < umbral


hay_alerta = revisar_umbral_calidad(EXACTITUD_CANCER)
print("Exactitud actual:", EXACTITUD_CANCER, "| umbral mínimo:", UMBRAL_EXACTITUD_MINIMA, "| ¿alerta?:", hay_alerta)
assert hay_alerta is False

# Simulamos un lote degradado (exactitud baja a propósito) para comprobar que la alerta SÍ se dispara
assert revisar_umbral_calidad(0.80) is True
print("La función detecta correctamente tanto el caso normal como el degradado.")

HERRAMIENTAS_REALES = [
    "DVC (clase 3): versionar este dataset y sus futuras variantes, con un remoto propio.",
    "MLflow (clases 4 y 8): registrar cada entrenamiento (semilla, hiperparámetros, métricas) y el modelo con alias candidato/producción/archivado.",
    "pandera (clase 6): un esquema de datos para las 30 columnas numéricas (rangos, nulos).",
    "FastAPI (clase 7): envolver predecir() en un endpoint real cuando el proyecto pase a producción.",
    "Docker + GitHub Actions (clase 8): empaquetar el servicio y automatizar tests antes de desplegar.",
    "Evidently (clase 11): vigilar el drift de las 30 columnas si la fuente de datos cambia con el tiempo.",
]
print("Herramientas que se usarían aquí en un proyecto real, SIN reimplementarlas (el criterio importa más que repetir código):")
for herramienta in HERRAMIENTAS_REALES:
    print("-", herramienta)

**¿Qué observas?** Con un dataset distinto y un modelo distinto, se repiten exactamente los mismos CINCO tipos de decisión de todo el curso: config + semilla explícita (clase 2), tests de datos y de modelo (clase 6), una función de servicio con validación mínima (clase 7) y un control de umbral de monitorización (clase 10) -- sin volver a montar DVC, MLflow, Docker ni FastAPI para este caso pequeño, pero sabiendo exactamente dónde encajaría cada uno.

---
## Limpieza final de la carpeta de trabajo

In [ ]:
shutil.rmtree(CARPETA)
print("Carpeta de trabajo eliminada:", CARPETA)
assert not os.path.exists(CARPETA)

---
## Resumen

- Un registro de auditoría legible se construye uniendo eventos de datos (DVC), experimentos (MLflow) y promociones (registro de modelos), y permite verificar principios de control de acceso como la separación de funciones.
- El coste de un sistema de ML en producción tiene al menos tres componentes con fórmulas distintas: entrenamiento (una vez), inferencia (por volumen) y almacenamiento (por tamaño) -- todas calculables en Python con constantes declaradas.
- El LLMOps comparte con el MLOps clásico la necesidad de trazar, versionar y vigilar, pero cambia la unidad de coste (tokens, no horas de cómputo) y la unidad de trazado (una cadena de pasos, con `mlflow.trace`, no un solo run).
- El mismo criterio de todo el curso (config explícita, tests mínimos, una función de servicio validada, un umbral de monitorización) se aplica igual de bien a un proyecto nuevo y pequeño que al proyecto de churn de las clases 2-11.